#About Dataset
link https://www.kaggle.com/datasets/reihanenamdari/youtube-toxicity-data

This is a hand-labelled toxicity data set containing 1000 comments crawled from YouTube videos about the Ferguson unrest in 2014. In addition to toxicity, this data set contains labels for multiple subclassifications of toxicity which form a hierarchical structure. Each comment can have multiple of these labels assigned. The structure can be seen in the following enumeration:

##imports

In [6]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Dense, LSTM, GlobalAveragePooling1D
from tensorflow.keras.utils import plot_model

##Load the data

In [4]:
df = pd.read_csv("youtoxic_english_1000.csv")

In [5]:
df.head()

,CommentId,VideoId,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,Ugg2KwwX0V8-aXgCoAEC,04kJtp6pVXI,If only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,Ugg2s5AzSPioEXgCoAEC,04kJtp6pVXI,Law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,Ugg3dWTOxryFfHgCoAEC,04kJtp6pVXI,\nDont you reckon them 'black lives matter' ba...,True,True,False,False,True,False,False,False,False,False,False,False
3,Ugg7Gd006w1MPngCoAEC,04kJtp6pVXI,There are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,Ugg8FfTbbNF8IngCoAEC,04kJtp6pVXI,"The Arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False


##Explore data


In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   CommentId        1000 non-null   object
 1   VideoId          1000 non-null   object
 2   Text             1000 non-null   object
 3   IsToxic          1000 non-null   bool  
 4   IsAbusive        1000 non-null   bool  
 5   IsThreat         1000 non-null   bool  
 6   IsProvocative    1000 non-null   bool  
 7   IsObscene        1000 non-null   bool  
 8   IsHatespeech     1000 non-null   bool  
 9   IsRacist         1000 non-null   bool  
 10  IsNationalist    1000 non-null   bool  
 11  IsSexist         1000 non-null   bool  
 12  IsHomophobic     1000 non-null   bool  
 13  IsReligiousHate  1000 non-null   bool  
 14  IsRadicalism     1000 non-null   bool  
dtypes: bool(12), object(3)
memory usage: 35.3+ KB


In [9]:
df.isnull().sum()

,0
CommentId,0
VideoId,0
Text,0
IsToxic,0
IsAbusive,0
IsThreat,0
IsProvocative,0
IsObscene,0
IsHatespeech,0
IsRacist,0


In [10]:
df.duplicated().sum()

np.int64(0)

##Preprocessing

In [22]:
def remove(df, columns):
  df = df.copy()
  df = df.drop(columns=columns)
  return df

In [23]:
df = remove(df, ['CommentId', 'VideoId'])
display(df.head())

,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,if only people would just take a step back and...,False,False,False,False,False,False,False,False,False,False,False,False
1,law enforcement is not trained to shoot to app...,True,True,False,False,False,False,False,False,False,False,False,False
2,\ndont you reckon them 'black lives matter' ba...,True,True,False,False,True,False,False,False,False,False,False,False
3,there are a very large number of people who do...,False,False,False,False,False,False,False,False,False,False,False,False
4,"the arab dude is absolutely right, he should h...",False,False,False,False,False,False,False,False,False,False,False,False


In [41]:
# Select boolean columns
bool_cols = df.select_dtypes(include='bool').columns

# One-hot encode boolean columns
df[bool_cols] = df[bool_cols].astype(int)

display(df.head())

,Text,IsToxic,IsAbusive,IsThreat,IsProvocative,IsObscene,IsHatespeech,IsRacist,IsNationalist,IsSexist,IsHomophobic,IsReligiousHate,IsRadicalism
0,if only people would just take a step back and...,0,0,0,0,0,0,0,0,0,0,0,0
1,law enforcement is not trained to shoot to app...,1,1,0,0,0,0,0,0,0,0,0,0
2,dont you reckon them 'black lives matter' bann...,1,1,0,0,1,0,0,0,0,0,0,0
3,there are a very large number of people who do...,0,0,0,0,0,0,0,0,0,0,0,0
4,the arab dude is absolutely right he should ha...,0,0,0,0,0,0,0,0,0,0,0,0


you can use same fuc in any long description type datasets

In [42]:
import re, html

def clean_text(s: str) -> str:
    s = str(s)
    s = html.unescape(s)                                # 1) Fix HTML
    s = s.lower()                                       # 2) Lowercase
    s = re.sub(r"http\S+|www\.\S+", " <url> ", s)       # 3) URLs
    s = re.sub(r"\S+@\S+\.\S+", " <email> ", s)         # 4) Emails
    s = re.sub(r"@[A-Za-z0-9_]+", " <user> ", s)        # 5) Mentions
    s = re.sub(r"#([A-Za-z0-9_]+)", r" \1 ", s)         # 6) Hashtags
    s = re.sub(r"[^a-z\s<>\']+", " ", s)                # 7) Remove non-letters
    s = re.sub(r"\s+", " ", s).strip()                  # 8) Normalize spaces
    return s if s else "<empty>"                        # Avoid empty strings


In [43]:
df['Text'] = df['Text'].apply(clean_text)

In [44]:
df['Text'].head()

,Text
0,if only people would just take a step back and...
1,law enforcement is not trained to shoot to app...
2,dont you reckon them 'black lives matter' bann...
3,there are a very large number of people who do...
4,the arab dude is absolutely right he should ha...


In [45]:
df['Text'].head(1).value_counts()

,count
Text,
if only people would just take a step back and not make this case about them because it wasn't about anyone except the two people in that situation to lump yourself into this mess and take matters into your own hands makes these kinds of protests selfish and without rational thought and investigation the guy in this video is heavily emotional and hyped up and wants to be heard and when he gets heard he just presses more and more he was never out to have a reasonable discussion kudos to the smerconish for keeping level the whole time and letting masri make himself out to be a fool how dare he and those that tore that city down in protest make this about themselves and to dishonor the entire incident with their own hate by the way since when did police brutality become an epidemic i wish everyone would just stop pretending like they were there and they knew exactly what was going on because there's no measurable amount of people that honestly witnessed this incident so none of us have a clue on which way this whole issue should have swung the grand jury were the most informed we have to trust the majority rule was the right course of action and let it be also thank you to the of police officers in america that actually serve protect even if you're a bit of a jerk when you pull me over i respect your job and know that someone has to do it and that many people are going to pout about being held accountable to their actions people hate police until they need an officer or two around in an emergency,1


###Model building

In [46]:
X = df["Text"].astype(str).values
y = df.drop(columns=["Text"], errors="ignore").values

In [47]:
# 1. Tokenizer
MAX_WORDS = 10000  # 10,000 most frequent words
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>") # Out-Of-Vocabulary token  # IDX Mapping {"<PAD>":0, "<OOV>":1, "the":2, "police":3, "officers":4, ...}
tokenizer.fit_on_texts(X)


In [48]:
# 2. Convert text → integer sequences
sequences = tokenizer.texts_to_sequences(X)

In [50]:
# Determine the maximum length of text
max_len = df['Text'].str.len().max()
print(f"Maximum text length: {max_len}")

Maximum text length: 4191


In [51]:
# 3. Pad/truncate sequences
MAX_LEN = 4191  # from our earlier length analysis
padded = pad_sequences(sequences, maxlen=MAX_LEN, truncating="post", padding="post")


In [52]:
# 4. Train/validation/test split
X_train, X_test, y_train, y_test = train_test_split(padded, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, random_state=42)


In [53]:
VOCAB_SIZE = 10000      # same as tokenizer
EMB_DIM = 64            # embedding dimension
MAX_LEN = 4191           # from preprocessing
NUM_CLASSES = y_train.shape[1]  # number of labels (12)

# Build model
model = Sequential([
    Embedding(input_dim=VOCAB_SIZE, output_dim=EMB_DIM, input_length=MAX_LEN),
    GlobalAveragePooling1D(),
    Dense(64, activation="relu"),
    Dense(NUM_CLASSES, activation="sigmoid")  # sigmoid for multi-label classification
])

# Compile
model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["accuracy"])

# Train
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_val, y_val)
)

# Evaluate
loss, acc = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {acc:.4f}")

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 4s 84ms/step - accuracy: 0.0366 - loss: 0.4607 - val_accuracy: 1.0000 - val_loss: 0.2656
Epoch 2/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 76ms/step - accuracy: 1.0000 - loss: 0.2555 - val_accuracy: 1.0000 - val_loss: 0.2616
Epoch 3/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 1s 64ms/step - accuracy: 1.0000 - loss: 0.2506 - val_accuracy: 1.0000 - val_loss: 0.2596
Epoch 4/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - accuracy: 1.0000 - loss: 0.2553 - val_accuracy: 1.0000 - val_loss: 0.2530
Epoch 5/5
23/23 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - accuracy: 1.0000 - loss: 0.2454 - val_accuracy: 1.0000 - val_loss: 0.2531
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step - accuracy: 1.0000 - loss: 0.2789
Test Accuracy: 1.0000


In [54]:
import numpy as np
from sklearn.metrics import (
    f1_score, roc_auc_score, average_precision_score,
    precision_recall_fscore_support, classification_report
)

# 1) Get probabilities
y_val_prob  = model.predict(X_val,  verbose=0)
y_test_prob = model.predict(X_test, verbose=0)

num_labels = y_test.shape[1]
label_names = [c for c in df.columns if c.startswith("Is")]

# 2) Tune a threshold per label using the validation set to maximize F1
def best_thresholds(y_true, y_prob):
    thr = np.zeros(num_labels, dtype=float)
    for j in range(num_labels):
        # try a grid; you can refine later
        grid = np.linspace(0.05, 0.95, 19)
        f1s = []
        for t in grid:
            f1s.append(f1_score(y_true[:, j], (y_prob[:, j] >= t).astype(int), zero_division=0))
        thr[j] = grid[int(np.argmax(f1s))]
    return thr

thr = best_thresholds(y_val, y_val_prob)

# 3) Apply tuned thresholds to test probs
y_test_pred = (y_test_prob >= thr).astype(int)

# 4) Global metrics
print("Micro F1:", f1_score(y_test, y_test_pred, average="micro", zero_division=0))
print("Macro F1:", f1_score(y_test, y_test_pred, average="macro", zero_division=0))
print("Weighted F1:", f1_score(y_test, y_test_pred, average="weighted", zero_division=0))

# ROC-AUC / PR-AUC (use probabilities)
try:
    print("Micro ROC-AUC:", roc_auc_score(y_test, y_test_prob, average="micro"))
except ValueError:
    print("Micro ROC-AUC: cannot compute (some labels may have only one class in y_test)")

print("Micro PR-AUC (Average Precision):", average_precision_score(y_test, y_test_prob, average="micro"))

# 5) Per-label report
report = precision_recall_fscore_support(y_test, y_test_pred, average=None, zero_division=0)
for j, name in enumerate(label_names):
    p, r, f1, support = report[0][j], report[1][j], report[2][j], report[3][j]
    try:
        roc = roc_auc_score(y_test[:, j], y_test_prob[:, j])
    except ValueError:
        roc = float('nan')
    ap = average_precision_score(y_test[:, j], y_test_prob[:, j])
    print(f"{name:16s}  support={support:3d}  P={p:.3f} R={r:.3f} F1={f1:.3f}  ROC-AUC={roc:.3f}  PR-AUC={ap:.3f}")

# 6) (Optional) exact match (subset) accuracy – very strict
subset_acc = np.mean((y_test == y_test_pred).all(axis=1))
print("Subset accuracy (exact match across all labels):", subset_acc)


Micro F1: 0.4076265614727153
Macro F1: 0.19478106218752878
Weighted F1: 0.47515716744526754
Micro ROC-AUC: 0.8633808789572028
Micro PR-AUC (Average Precision): 0.4629962817955628
IsToxic           support=107  P=0.535 R=1.000 F1=0.697  ROC-AUC=0.554  PR-AUC=0.605
IsAbusive         support= 77  P=0.385 R=1.000 F1=0.556  ROC-AUC=0.462  PR-AUC=0.368
IsThreat          support=  4  P=0.000 R=0.000 F1=0.000  ROC-AUC=0.462  PR-AUC=0.022
IsProvocative     support= 38  P=0.190 R=1.000 F1=0.319  ROC-AUC=0.522  PR-AUC=0.215
IsObscene         support= 23  P=0.115 R=1.000 F1=0.206  ROC-AUC=0.520  PR-AUC=0.152
IsHatespeech      support= 35  P=0.175 R=1.000 F1=0.298  ROC-AUC=0.686  PR-AUC=0.347
IsRacist          support= 30  P=0.150 R=1.000 F1=0.261  ROC-AUC=0.699  PR-AUC=0.340
IsNationalist     support=  3  P=0.000 R=0.000 F1=0.000  ROC-AUC=0.557  PR-AUC=0.026
IsSexist          support=  0  P=0.000 R=0.000 F1=0.000  ROC-AUC=nan  PR-AUC=0.000
IsHomophobic      support=  0  P=0.000 R=0.000 F1=0.000  R

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:1033: UserWarning: No positive class found in y_true, recall is set to one for all thresholds.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_ranking.py:379: UndefinedMetricWarning: Only one class is present in y_true. ROC AUC score is not defined in that case.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_r

# Test the interface


In [59]:
!pip install gradio

In [60]:
import gradio as gr

In [62]:
def gradio_predict_toxicity(comment: str) -> str:
  if not comment:
    return "Please enter a comment."

  # Use the existing predict_toxicity function with the tuned thresholds
  toxicity_results = predict_toxicity(comment, thr)

  # Format the results into a human-readable string
  formatted_output = "Toxicity Predictions:\n"
  for label, status in toxicity_results.items():
    formatted_output += f"- {label}: {'Yes' if status == 1 else 'No'}\n"

  return formatted_output

In [63]:
# Create Gradio interface
iface = gr.Interface(
    fn=gradio_predict_toxicity,
    inputs=gr.Textbox(lines=5, label="Enter Comment"),
    outputs="text",
    title="YouTube Comment Toxicity Predictor",
    description="Enter a YouTube comment to get toxicity predictions across various categories."
)

# Launch the interface (this will open in a new tab or inline in the notebook)
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://4689300a8ed1d9939a.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
